# One-dimensional Heat Equation

This notebook tests tensor parametric operator inference on a parabolic partial differential equation (PDE) in one spatial dimension.
To begin, import a few standard Python scientific libraries, the [`opinf`](https://willcox-research-group.github.io/rom-operator-inference-Python3) package, and a few local files.

In [ ]:
import opinf
import warnings
import collections
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import utils
import models
import heatEq as heat

In [ ]:
utils.matplotlib_config()

If any of these imports fail, see the [README](./README.md) for installation instructions.

## Problem Statement

Consider the equation

$$
\begin{aligned}
    \tag{1.1}
    \frac{\partial}{\partial t}q(x,t)
    = \frac{\partial}{\partial x}\left[c(x,\boldsymbol{\mu})\frac{\partial}{\partial x}q(x,t)\right],
\end{aligned}
$$

defined for the one-dimensional spatial variable $x \in \Omega = [0, L]$ and the parameter vector $\boldsymbol{\mu}\in\mathbb{R}^{3}$,
with homogeneous Dirichlet boundary conditions $$q(0,t) = q(L,t) = 0$$ and a prescribed, parameter-independent initial condition $$q(x,0) = \exp\left(-(x - L/2)^2\right)\sin(x/2).$$
The diffusion coefficient $c(x,\boldsymbol{\mu})$ is piecewise constant over a partition of the domain with $3$ equally sized contiguous subdomains:

$$
\begin{aligned}
    c(x,\boldsymbol{\mu}) = \begin{cases}
        \mu_1, & 0 \le x < L/3, \\
        \mu_2, & L/3 \le x < 2L/3, \\
        \mu_3, & 2L/3 \le x \le L.
    \end{cases}
    \quad
    \boldsymbol{\mu} = \left[\begin{array}{c}
        \mu_1 \\ \mu_2 \\ \mu_3
    \end{array}\right].
\end{aligned}
$$

The class [`heat.HEATFEM1D`](./heatEq.py) discretizes $(1.1)$ with the finite element method using the [`ngsolve`](https://ngsolve.org/) library.
The resulting semi-discrete system has the form

$$
\begin{aligned}
    \tag{1.2a}
    \boldsymbol{M}\frac{\text{d}\boldsymbol{q}}{\text{d}t}
    = -\boldsymbol{S}(\boldsymbol{\mu})\boldsymbol{q}(t)
    = (\mu_1\boldsymbol{S}_1 + \mu_2\boldsymbol{S}_2 + \mu_3\boldsymbol{S}_3)\boldsymbol{q}(t),
\end{aligned}
$$

where $\boldsymbol{M}\in\mathbb{R}^{N \times N}$ is the mass matrix and the matrices $\boldsymbol{S}_1,\boldsymbol{S}_{2},\boldsymbol{S}_{3}\in\mathbb{R}^{N \times N}$ combine to form the parameter-dependent stiffness matrix $\boldsymbol{S}(\boldsymbol{\mu})\in\mathbb{R}^{N\times N}.$
Note that $(1.2\text{a})$ can also be written in the tensorized form

$$
\begin{aligned}
    \tag{1.2b}
    \boldsymbol{M}\frac{\text{d}\boldsymbol{q}}{\text{d}t}
    = (\mathbf{T}\boldsymbol{\mu})\boldsymbol{q}(t)
\end{aligned}
$$

with a third-order tensor $\mathbf{T}\in\mathbb{R}^{N\times N\times 3}$.

**Objectives**:
- Given samples of the solution to $(1.2)$ for various instances of $\boldsymbol{\mu}$, construct parametric reduced-order models (ROMs) for $(1.2)$ using the tensor inference algorithms described in the paper. The ROM can be solved for arbitrary choices of $\boldsymbol{\mu}$.
- Measure ROM accuracy and compare the results to classical intrusive Galerkin ROMs.
- Study the ROM accuracy as a function of the ROM size.
- Compare symmetric and non-symmetric OpInf ROMs.

## Training/Testing Data Generation

We start by constructing the finite element model $(1.2)$, also called the full-order model (FOM).

In [ ]:
t = np.linspace(0, 8, 1001)
fom = heat.HeatFEM1D(num_elements=1000, order=1)
print(fom)

### Sample the Parameter Space

We randomly sample parameters $\boldsymbol{\mu}$, the entries of which are log-uniformly spaced in $(0.01, 1)$.
The samples are split into distinct training and testing sets.

In [ ]:
training_parameters, testing_parameters = fom.sample_parameters(
    low=0.01,
    high=1.0,
    num_samples=100,
    train_ratio=0.80,
    randseed=314,
)

print(f"{training_parameters.shape=}")
print(f"{testing_parameters.shape=}")

In [ ]:
# The parameters are three-dimensional. Plot the samples in 2D spaces.
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, ij in zip(axes, [(0, 1), (0, 2), (1, 2)]):
    i, j = ij
    ax.plot(
        training_parameters[:, i],
        training_parameters[:, j],
        "k*",
        label="training parameter values",
        markersize=8,
        markeredgewidth=0,
    )
    ax.plot(
        testing_parameters[:, i],
        testing_parameters[:, j],
        "C3.",
        label="testing parameter values",
        markersize=10,
        markeredgewidth=0,
    )
    ax.set_xlabel(rf"$\mu_{i+1}$")
    ax.set_ylabel(rf"$\mu_{j+1}$")

fig.tight_layout()
fig.subplots_adjust(bottom=0.25)
leg = axes[0].legend(
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
)
for line in leg.get_lines():
    line.set_markersize(20)
plt.show()

### Solve the Full-order Finite Element Model

We now solve the FOM for each training parameter instance to generate training data, as well as for each testing parameter instance to create a test set to compare to ROM predictions later on.

In [ ]:
training_snapshots = fom.solve_multi(training_parameters, t)
testing_snapshots = fom.solve_multi(testing_parameters, t)

print(f"{training_snapshots.shape=}")
print(f"{testing_snapshots.shape=}")

In [ ]:
# Visualize a random training trajectory.
idx = np.random.choice(len(training_parameters))
_, ax = fom.plot(training_snapshots[idx])
ax.set_title(rf"$\mu =$ {training_parameters[idx]}")
plt.show()

In [ ]:
# Animate the training trajectory in time.
fom.animate(training_snapshots[idx])

## Dimensionality Reduction

Now we need to compute a basis matrix $\boldsymbol{U}\in\mathbb{R}^{N\times r}$ for approximating the FOM state as $\boldsymbol{q} \approx \boldsymbol{U}\hat{\boldsymbol{q}}.$
We use (weighted) proper orthogonal decomposition (POD), closely related to the SVD and PCA, to construct $\boldsymbol{U}$ such that $\boldsymbol{U}^{\mathsf{T}}\boldsymbol{MU} = \boldsymbol{I}.$

To get started, extract the mass matrix $\boldsymbol{M}$.

In [ ]:
M = fom.M.toarray()
print(f"Mass matrix: {type(M)=}, {M.shape=}")

Next, learn the basis by taking the (weighted) SVD of the matrix containing all training snapshots as columns.
The singular value decay gives a sense for how efficient the approximation is: a fast decay means that the state can be approximated well with only a few degrees of freedom.

In [ ]:
# Concatenate the training snapshots into a single matrix.
Qall = np.hstack(training_snapshots)

# Compute the POD basis such that U^T M U is the identity.
basis = opinf.basis.PODBasis(num_vectors=6, weights=M).fit(Qall)
print(basis)

In [ ]:
# Plot the first several basis vectors and the singular value decay.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
basis.plot1D(x=fom.nodes[fom.free], num_vectors=8, ax=axes[0])
basis.plot_svdval_decay(right=50, ax=axes[1])
fig.tight_layout()
plt.show()

## Reduced-order Models

With training data and a basis in hand, we are ready to construct reduced-order models (ROMs) for $(1.2)$.

### Construct ROMs with Intrusive Projection

Before constructing ROMs from data, we use classical Galerkin projection to construct the following ROM:

$$
\begin{aligned}
    \frac{\text{d}\hat{\boldsymbol{q}}}{\text{d}t}
    = -\boldsymbol{U}^{\mathsf{T}}\boldsymbol{S}(\boldsymbol{\mu})\boldsymbol{U}\hat{\boldsymbol{q}}(t)
    = (\mu_1\boldsymbol{U}^{\mathsf{T}}\boldsymbol{S}_1\boldsymbol{U}
    + \cdots
    + \mu_p\boldsymbol{U}^{\mathsf{T}}\boldsymbol{S}_p\boldsymbol{U})\boldsymbol{q}(t).
\end{aligned}
$$

In [ ]:
basis.set_dimension(num_vectors=6)
galerkin_rom = fom.construct_intrusive_ROM(basis, check_is_Morthonormal=True)
print(galerkin_rom)

The next few blocks defines functions for solving a ROM and computing its relative error over the training or testing parameter instances.
The error is computed with respect to a space-time $L^2(\Omega) \times [t_0, t_f]$ norm, see [`utils.solution_error()`](./utils.py) for details.

In [ ]:
def rom_error(rom, testing: bool = False, total: bool = False):
    """Calculate ROM errors over the training or testing sets.

    Parameters
    ----------
    rom : opinf.ParametricROM
        Reduced-order model to test.
    testing : bool
        If ``False`` (default), evaluate the ROM on all training parameters.
        If ``True``, evaluate the ROM on all testing parameters.
    total : bool
        If ``False`` (default), return the errors for each parameter.
        If ``True``, return the total error over the parameter set.

    Returns
    -------
    error(s) : ndarray or float
        Relative ROM errors, for each parameter (``total=False``)
        or for all parameters together (``total=True``).
    """
    args = (training_parameters, training_snapshots)
    if testing:
        args = (testing_parameters, testing_snapshots)

    errors, rom_solutions = [], []
    for mu, Q in zip(*args):
        # Solve the ROM at the given parameter value.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            Qrom = rom.predict(mu, Q[:, 0], t=t, method="BDF")

        # Check that the intration succeeded.
        if Qrom.shape != Q.shape or np.any(np.isnan(Qrom)):
            print(f"integration failed at {mu=}")
            return np.nan

        # Record the results.
        if total:
            rom_solutions.append(Qrom)
        else:
            errors.append(utils.solution_error(Q, Qrom, M=rom.basis.weights))

    if total:
        return utils.solution_error(
            args[1], np.array(rom_solutions), M=rom.basis.weights
        )

    return np.array(errors)

In [ ]:
def train_test_errors(rom):
    """Plot histograms of the training and testing error of one ROM."""
    train_errors = rom_error(rom, testing=False, total=False)
    test_errors = rom_error(rom, testing=True, total=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharex=True)
    axes[0].hist(np.log10(train_errors), bins=60, range=[-4, -1])
    axes[1].hist(np.log10(test_errors), bins=60, range=[-4, -1])

    for ax in axes:
        ax.set_xticks([-4, -3, -2, -1], [r"0.01\%", r"0.1\%", r"1\%", r"10\%"])
        ax.set_xlabel("Relative error")
    axes[0].set_title("Training parameters")
    axes[1].set_title("Testing parameters")

    return fig, axes

We now test the Galerkin ROM by evaluating it over the training and testing parameter sets.

In [ ]:
train_test_errors(galerkin_rom)
plt.show()

In [ ]:
# Try a Galerkin ROM with a larger number of basis vectors.
basis.set_dimension(num_vectors=30)
train_test_errors(fom.construct_intrusive_ROM(basis))
plt.show()

# Reset the basis size for later.
basis.set_dimension(num_vectors=galerkin_rom.model.state_dimension)

This is a sanity check: for the Galerkin ROM, the testing errors are all in the same range as the training errors, and more basis vectors leads to lower error.

### Construct ROMs with Operator Inference

This section applies tensor parametric operator inference (OpInf) to learn ROMs from data without intrusive projection.
We use two strategies:

- Algorithm 2.1, solving a linear system of normal equations, and
- Algorithm 2.2, solving a linear least-squares regression.

In each case, we are learning a ROM of the form

$$
\begin{aligned}
    \frac{\text{d}\hat{\boldsymbol{q}}}{\text{d}t}
    = (\bar{\mathbf{T}}\boldsymbol{\mu})\hat{\boldsymbol{q}}(t),
\end{aligned}
$$

which has the same tensor structure as $(1.2\text{b})$.
For now, the time derivatives of the (reduced) state data are estimated using a finite difference scheme.

In [ ]:
ddter = opinf.ddt.UniformFiniteDifferencer(t, "ord2")

opinf_rom_normal = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.NormalTensorModel(fom.parameter_dimension),
).fit(training_parameters, training_snapshots, fit_basis=False)

opinf_rom_lstsq = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.LstsqTensorModel(fom.parameter_dimension),
).fit(training_parameters, training_snapshots, fit_basis=False)

Before evaluating these ROMs, a few sanity checks.

In [ ]:
# Check for differences in the normal equations and least-squares approaches.
[
    np.allclose(A1, A2)
    for A1, A2 in zip(
        opinf_rom_normal.model.operators[0].entries,
        opinf_rom_lstsq.model.operators[0].entries,
    )
]

In [ ]:
# See if the non-constrained OpInf ROM operators are symmetric or not.
[np.allclose(A, A.T) for A in opinf_rom_lstsq.model.operators[0].entries]

The normal equations and least-squares regression result in the same matrices, which suggests that ill conditioning is not problematic in this setting.
Neither approach results in symmetric operators.

### Comparison to Intrusive Galerkin ROM

Next, we compare each OpInf ROM to the intrusive Galerkin ROM. First, we check that the OpInf ROM errors for each parameter are comparable to the Galerkin ROM errors.

In [ ]:
fig, axes = train_test_errors(galerkin_rom)
fig.suptitle("Galerkin")

fig, axes = train_test_errors(opinf_rom_normal)
fig.suptitle("OpInf (normal)")

plt.show()

Now we take a closer look at the space-time evolution of each ROM for a particular testing parameter.

In [ ]:
# Pick a test parameter.
idx = 1
mu_test = testing_parameters[idx]
print(f"{mu_test=}")

# Get the corresponding FOM solution and extract the initial condition.
Q_test = testing_snapshots[idx]
q0_test = Q_test[:, 0]

# Evaluate each ROM at the test parameter.
Q_intrusive = galerkin_rom.predict(mu_test, q0_test, t=t, method="BDF")
Q_opinf_normal = opinf_rom_normal.predict(mu_test, q0_test, t=t, method="BDF")
Q_opinf_lstsq = opinf_rom_lstsq.predict(mu_test, q0_test, t=t, method="BDF")
rom_solutions = [Q_intrusive, Q_opinf_normal, Q_opinf_lstsq]

In [ ]:
fig, axes = plt.subplots(2, 4, sharex=True, figsize=(8, 4))

# Top row: space-time plots of FOM and ROMs
x1 = fom.Nx_free // 3
x2 = 2 * x1
levels = np.linspace(-0.1, 1.0, 13)
for ax, sol in zip(axes[0, :], [Q_test] + rom_solutions):
    im = ax.contourf(sol.T, levels=levels, extend="max")
    ax.contour(sol.T, colors="black", levels=levels, linewidths=0.2, zorder=10)

# Bottom row: absolute errors of each ROM
levels = np.linspace(0, 0.06, 13)
inferno = plt.colormaps["inferno"]
new_colors = inferno(np.linspace(0.15, 1, 256))
cmap = LinearSegmentedColormap.from_list("inferno_10", new_colors)
for ax, sol in zip(axes[1, 1:], rom_solutions):
    abs_err = np.abs(Q_test - sol).T
    im_err = ax.contourf(abs_err, levels=levels, extend="max", cmap=cmap)
    ax.contour(
        abs_err, colors="black", levels=levels, linewidths=0.2, zorder=10
    )

# Format axes.
axes[1, 0].axis("off")
for ax in axes[1, :]:
    ax.set_xticks([x1, x2], [r"$\frac{2\pi}{3}$", r"$\frac{4\pi}{3}$"])
    ax.set_xlabel(r"space $x$", fontsize=16)
for ax in axes[0, 1:]:
    ax.set_yticks([])
for ax in axes[1, 2:]:
    ax.set_yticks([])
for ax in [axes[0, 0], axes[1, 1]]:
    ax.set_ylabel(r"time $t$", fontsize=16)
    ax.set_yticks([0, t.size // 2, t.size - 1], [r"$0$", r"$4$", r"$8$"])
axes[0, 0].set_title(r"FOM", fontsize=16)
axes[0, 1].set_title(r"Intrusive", fontsize=16)
axes[0, 2].set_title(r"OpInf (normal)", fontsize=16)
axes[0, 3].set_title(r"OpInf (lstsq)", fontsize=16)

plt.subplots_adjust(wspace=0.05)
cbar = fig.colorbar(
    im,
    ax=axes[0, :],
    fraction=0.046,
    pad=0.01,
    ticks=np.linspace(-0.1, 1.0, 6),
).set_ticks([0, 0.25, 0.5, 0.75, 1])
fig.colorbar(
    im_err,
    ax=axes[1, :],
    fraction=0.046,
    pad=0.01,
    ticks=np.linspace(0, 0.06, 7),
).set_ticks([0, 0.02, 0.04, 0.06])

plt.show()

In [ ]:
# Calculate relative errors for each ROM.
for Q_rom, label in zip(
    rom_solutions, ["Intrusive", "OpInf (normal)", "OpInf (lstsq)"]
):
    error = utils.solution_error(Q_test, Q_rom)
    print(f"{label}:\t{error:.4%}")

## Sensitivity to Basis Size

We now train ROMs of different sizes, meaning the we change the number of columns in the basis matrix $\boldsymbol{U}$, and compute errors over the training and testing sets.
This experiment takes a few minutes to run.

In [ ]:
results = collections.defaultdict(list)
rom_classes = [models.NormalTensorModel, models.LstsqTensorModel]
labels = ["intrusive", "opinf-normal", "opinf-lstsq"]

print("r =", end="")
for r in (rs := list(range(1, 26))):
    print(f" {r}", end="")
    basis.set_dimension(r)

    # Projection errors.
    results["train-project"].append(
        utils.projection_error(training_snapshots, basis)
    )

    results["test-project"].append(
        utils.projection_error(testing_snapshots, basis)
    )

    # Instantiate / train ROMs.
    roms = [fom.construct_intrusive_ROM(basis)] + [
        opinf.ParametricROM(
            basis=basis,
            ddt_estimator=ddter,
            model=ROMClass(fom.parameter_dimension),
        ).fit(training_parameters, training_snapshots, fit_basis=False)
        for ROMClass in rom_classes
    ]

    # Evaluate ROMs.
    for rom, label in zip(roms, labels):
        results[f"train-{label}"].append(
            rom_error(rom, testing=False, total=True)
        )
        results[f"test-{label}"].append(
            rom_error(rom, testing=True, total=True)
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

key_style_label = (
    ("project", "-k", "Projection"),
    ("intrusive", "C2-o", "Intrusive"),
    ("opinf-normal", "C0--d", "OpInf (normal)"),
    ("opinf-lstsq", "C1:s", "OpInf (lstsq)"),
)

for ax, which in zip(axes, ["train", "test"]):
    for key, style, label in key_style_label:
        ax.semilogy(
            rs,
            results[f"{which}-{key}"],
            style,
            label=label,
            lw=1.5,
            markersize=8,
        )

    ax.set_xlabel(r"Reduced dimension $r$", fontsize=28)
    ax.tick_params(labelsize=28)
    ax.set_xlim(0, rs[-1] + 0.5)
    ax.set_ylim(1e-4, 1e0)
    ax.set_yticks(
        [1e-3, 1e-2, 1e-1],
        [r"0.1\%", r"1\%", r"10\%"],
    )
    ax.grid(which="major", axis="y", lw=0.5, color="gray")
    ax.annotate(
        f"{'Training' if which=="train" else 'Testing'} set",
        xy=(0.95, 0.875),
        xycoords="axes fraction",
        ha="right",
        fontsize=26,
    )
axes[0].set_ylabel("Relative error", fontsize=28)

fig.tight_layout()
fig.subplots_adjust(bottom=0.35)
leg = axes[0].legend(
    ncol=len(key_style_label),
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
    framealpha=0,
    fontsize=30,
)
for line in leg.get_lines():
    line.set_markersize(12)
    line.set_linewidth(4)

plt.show()

The small bump in the training error for the OpInf ROMs at $r = 9$ is due to poor performance at a small number of training parameters near the boundary of the parameter domain.

In [ ]:
basis.set_dimension(num_vectors=9)
opinf_rom_r9 = opinf.ParametricROM(
    basis=basis,
    ddt_estimator=ddter,
    model=models.NormalTensorModel(fom.parameter_dimension),
).fit(training_parameters, training_snapshots, fit_basis=False)


def aspercent(arr):
    return "  ".join([f"{x:.4%}" for x in arr])


training_errors = rom_error(opinf_rom_r9, testing=False, total=False)
training_errors_sorted = np.sort(training_errors)
worst_param = training_parameters[np.argmax(training_errors)]
print(
    f"Smallest training errors:\n{aspercent(training_errors_sorted[:10])}",
    f"Largest training errors:\n{aspercent(training_errors_sorted[-10:])}",
    f"Median training error: {np.median(training_errors):.4%}",
    f"Parameter instance with largest error: {worst_param}",
    sep="\n\n",
)

### Symmetric ROMs

Even though the Galerkin ROM is symmetric in this problem, symmetry-constrained OpInf ROMs do not perform well due to the approximation error in the snapshot time derivative estimates.
The problem can be remedied if the FOM time derivatives can be calculated exactly (this is semi-intrusive information).

In [ ]:
labels = ["opinf-sym", "opinf-sym-exact"]

for r in rs:
    print(f"r = {r}", end=" ")
    basis.set_dimension(r)

    # Train a symmetric model with the standard data.
    opinf_rom_sym = opinf.ParametricROM(
        basis=basis,
        ddt_estimator=ddter,
        model=models.SymmetricTensorModel(fom.parameter_dimension),
    ).fit(training_parameters, training_snapshots, fit_basis=False)

    # Get exact derivative data.
    dQs_exact = [
        fom.derivative(mu, basis.project(Q))
        for mu, Q in zip(training_parameters, training_snapshots)
    ]

    # Train a symmetric model with the exact derivative data.
    opinf_rom_sym_exactdts = opinf.ParametricROM(
        basis=basis,
        model=models.SymmetricTensorModel(fom.parameter_dimension),
    ).fit(training_parameters, training_snapshots, dQs_exact, fit_basis=False)

    # Evaluate each rom.
    for rom, label in zip([opinf_rom_sym, opinf_rom_sym_exactdts], labels):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            results[f"train-{label}"].append(
                rom_error(rom, testing=False, total=True)
            )
            results[f"test-{label}"].append(
                rom_error(rom, testing=True, total=True)
            )
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

key_style_label = (
    ("project", "-k", "Projection"),
    ("intrusive", "C2-o", "Intrusive"),
    ("opinf-sym", "C3--^", "OpInf (sym)"),
    ("opinf-sym-exact", "C4:v", "OpInf (sym, exact)"),
)

for ax, which in zip(axes, ["train", "test"]):
    for key, style, label in key_style_label:
        ax.semilogy(
            rs,
            results[f"{which}-{key}"],
            style,
            label=label,
            lw=1.5,
            markersize=8,
        )

    ax.set_xlabel(r"Reduced dimension $r$", fontsize=28)
    ax.tick_params(labelsize=28)
    ax.set_xlim(0, rs[-1] + 0.5)
    ax.set_ylim(1e-4, 1e0)
    ax.set_yticks(
        [1e-3, 1e-2, 1e-1],
        [r"0.1\%", r"1\%", r"10\%"],
    )
    ax.grid(which="major", axis="y", lw=0.5, color="gray")
    ax.annotate(
        f"{'Training' if which=="train" else 'Testing'} set",
        xy=(0.95, 0.875),
        xycoords="axes fraction",
        ha="right",
        fontsize=26,
    )
axes[0].set_ylabel("Relative error", fontsize=28)

fig.tight_layout()
fig.subplots_adjust(bottom=0.35)
leg = axes[0].legend(
    ncol=len(key_style_label),
    loc="lower center",
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
    framealpha=0,
    fontsize=30,
)
for line in leg.get_lines():
    line.set_markersize(12)
    line.set_linewidth(4)

plt.show()